# GridSentinel: End-to-End Cyber-Physical Security ML Pipeline

This notebook demonstrates the end-to-end machine learning pipeline used by the GridSentinel backend to detect False Data Injections (FDI), AGC Scaling attacks, and multi-vector threats leveraging both traditional ensemble methods and Deep Learning (LSTM).

It covers:
1. **Grid Telemetry Synthesis**: Simulating IEEE 14-bus test case measurements (Voltages, Phase Angles, Branch Flows).
2. **Exploratory Data Analysis (EDA)**: Correlation matrices and attack distributions.
3. **Traditional ML (Ensemble Detectors)**: Random Forest & Gradient Boosting analysis (Confusion Matrices, ROC, PR Curves, Feature Importance).
4. **Deep Learning (LSTM)**: Temporal anomaly sequence processing, training curves, and evaluation.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import classification_report, confusion_matrix, f1_score, roc_curve, auc, precision_recall_curve

import warnings
warnings.filterwarnings('ignore')

# Set aesthetic parameters for premium plots
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'sans-serif'

np.random.seed(42)
torch.manual_seed(42)

## 1. Grid Telemetry Synthesis

In the GridSentinel app, high-resolution **Phasor Measurement Units (PMUs)** (IEEE C37.118) output 20 key features based on the IEEE 14-bus framework:
- `F00 - F05`: Nodal Voltages (steady around 1.0 p.u.)
- `F06 - F11`: Phase Angles (relative to slack bus)
- `F12 - F19`: Active Branch/Line Flows

We synthesize 10,000 baseline observations, and then inject realistic False Data Injection (FDI) and AGC Scaling attacks.

In [ ]:
N_SAMPLES = 10000
N_FEATURES = 20

def generate_baseline(n):
    # Base measurements slightly oscillating over time to simulate demand curve
    t = np.linspace(0, 100, n)
    data = np.zeros((n, N_FEATURES))
    
    # Voltages ~ 1.0
    data[:, 0:6] = 1.0 + 0.01 * np.sin(t[:, np.newaxis] * 0.5) + np.random.normal(0, 0.005, (n, 6))
    # Angles ~ localized around 0
    data[:, 6:12] = 0.05 * np.cos(t[:, np.newaxis] * 0.8) + np.random.normal(0, 0.01, (n, 6))
    # Flows ~ nominal
    data[:, 12:20] = 0.5 + 0.1 * np.sin(t[:, np.newaxis] * 0.3) + np.random.normal(0, 0.015, (n, 8))
    return data

X_normal = generate_baseline(N_SAMPLES)
y_normal = np.zeros(N_SAMPLES)

# Attack 1: FDI on Phase Angles and specific lines (Features 6, 7, 12, 13)
X_fdi = generate_baseline(3000)
X_fdi[:, [6, 7]] += np.random.uniform(0.1, 0.3, (3000, 2))  # Phase shift
X_fdi[:, [12, 13]] *= np.random.uniform(0.5, 0.8, (3000, 2)) # Artificial line drop
y_fdi = np.ones(3000)

# Attack 2: AGC Ramp Attack (Slowly drifting all frequencies/angles)
X_agc = generate_baseline(2000)
ramp = np.linspace(0, 0.15, 2000)[:, np.newaxis]
X_agc[:, 6:12] += ramp
y_agc = np.ones(2000)

# Combine Dataset
X = np.vstack([X_normal, X_fdi, X_agc])
y = np.concatenate([y_normal, y_fdi, y_agc])

feature_names = [f'V_Mag_B{i+1}' for i in range(6)] + [f'P_Ang_B{i+1}' for i in range(6)] + [f'Flow_L{i+1}' for i in range(8)]
df = pd.DataFrame(X, columns=feature_names)
df['Label'] = y

print(f"Total Samples: {len(df)} | Normal: {(y==0).sum()} | Attack (FDI+AGC): {(y==1).sum()}")

## 2. Exploratory Data Analysis (EDA)

Before training, we analyze feature distributions and correlations to understand how attacks distort the physical grid relationships (the covariance matrix).

In [ ]:
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
sns.kdeplot(df[df.Label == 0]['P_Ang_B1'], label='Normal', fill=True, color='#10b981')
sns.kdeplot(df[df.Label == 1]['P_Ang_B1'], label='Attack', fill=True, color='#ef4444')
plt.title('Distribution: Phase Angle 1 (P_Ang_B1)')
plt.xlabel('Phase Angle (Rads)')
plt.legend()

plt.subplot(1, 2, 2)
sns.kdeplot(df[df.Label == 0]['Flow_L1'], label='Normal', fill=True, color='#10b981')
sns.kdeplot(df[df.Label == 1]['Flow_L1'], label='Attack', fill=True, color='#ef4444')
plt.title('Distribution: Line Flow 1 (Flow_L1)')
plt.xlabel('Active Power Flow (p.u.)')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Correlation Matrix Heatmap
plt.figure(figsize=(12, 10))
corr = df.drop('Label', axis=1).corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap='coolwarm', center=0, vmax=1.0, vmin=-1.0, square=True, linewidths=.5, cbar_kws={"shrink": .8})
plt.title('Feature Correlation Matrix (Normal Operational States)')
plt.show()

## 3. Traditional ML: Ensemble Detectors

In the GridSentinel app, `RandomForestClassifier` and `GradientBoostingClassifier` act as robust, low-latency detectors for instantaneous structural anomalies.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(df.drop('Label', axis=1), df['Label'], test_size=0.2, random_state=42, stratify=df['Label'])

# Scale for Gradient Boosting and Neural Nets later
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Training Random Forest...")
rf_clf = RandomForestClassifier(n_estimators=100, max_depth=12, random_state=42, n_jobs=-1)
rf_clf.fit(X_train, y_train)

print("Training Gradient Boosting...")
gb_clf = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=5, random_state=42)
gb_clf.fit(X_train_scaled, y_train)

y_prob_rf = rf_clf.predict_proba(X_test)[:, 1]
y_prob_gb = gb_clf.predict_proba(X_test_scaled)[:, 1]
y_pred_rf = (y_prob_rf > 0.5).astype(int)

In [ ]:
# 3A: Confusion Matrices
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
sns.heatmap(confusion_matrix(y_test, y_pred_rf), annot=True, fmt='d', cmap='Blues', ax=ax[0], xticklabels=['Normal', 'Attack'], yticklabels=['Normal', 'Attack'])
ax[0].set_title('Random Forest Confusion Matrix')
ax[0].set_ylabel('Actual')
ax[0].set_xlabel('Predicted')

sns.heatmap(confusion_matrix(y_test, (y_prob_gb > 0.5).astype(int)), annot=True, fmt='d', cmap='Purples', ax=ax[1], xticklabels=['Normal', 'Attack'], yticklabels=['Normal', 'Attack'])
ax[1].set_title('Gradient Boosting Confusion Matrix')
ax[1].set_ylabel('Actual')
ax[1].set_xlabel('Predicted')
plt.show()

In [ ]:
# 3B: ROC & Precision-Recall Curves
fpr_rf, tpr_rf, _ = roc_curve(y_test, y_prob_rf)
fpr_gb, tpr_gb, _ = roc_curve(y_test, y_prob_gb)
roc_auc_rf = auc(fpr_rf, tpr_rf)
roc_auc_gb = auc(fpr_gb, tpr_gb)

precision_rf, recall_rf, _ = precision_recall_curve(y_test, y_prob_rf)
precision_gb, recall_gb, _ = precision_recall_curve(y_test, y_prob_gb)
pr_auc_rf = auc(recall_rf, precision_rf)
pr_auc_gb = auc(recall_gb, precision_gb)

fig, ax = plt.subplots(1, 2, figsize=(14, 6))
# ROC
ax[0].plot(fpr_rf, tpr_rf, color='darkorange', lw=2, label=f'Random Forest (AUC = {roc_auc_rf:.4f})')
ax[0].plot(fpr_gb, tpr_gb, color='indigo', lw=2, label=f'Gradient Boosting (AUC = {roc_auc_gb:.4f})')
ax[0].plot([0, 1], [0, 1], color='gray', lw=2, linestyle='--')
ax[0].set_xlim([0.0, 1.0])
ax[0].set_ylim([0.0, 1.05])
ax[0].set_xlabel('False Positive Rate')
ax[0].set_ylabel('True Positive Rate')
ax[0].set_title('Receiver Operating Characteristic (ROC)')
ax[0].legend(loc="lower right")

# PR
ax[1].plot(recall_rf, precision_rf, color='darkorange', lw=2, label=f'Random Forest (AUC = {pr_auc_rf:.4f})')
ax[1].plot(recall_gb, precision_gb, color='indigo', lw=2, label=f'Gradient Boosting (AUC = {pr_auc_gb:.4f})')
ax[1].set_xlim([0.0, 1.0])
ax[1].set_ylim([0.0, 1.05])
ax[1].set_xlabel('Recall')
ax[1].set_ylabel('Precision')
ax[1].set_title('Precision-Recall Curve')
ax[1].legend(loc="lower left")

plt.show()

In [ ]:
# 3C: Feature Importance
importances = rf_clf.feature_importances_
indices = np.argsort(importances)[::-1]

plt.figure(figsize=(12, 6))
plt.title("Random Forest: Global Feature Importances (Top 12)")
plt.bar(range(12), importances[indices][:12], color='teal', align="center")
plt.xticks(range(12), [feature_names[i] for i in indices[:12]], rotation=45)
plt.xlim([-1, 12])
plt.ylabel('Gini Importance')
plt.tight_layout()
plt.show()

## 4. Deep Learning: Temporal Anomaly Detection with LSTM

While RF and GB are great, they process row-by-row. Cyber-physical attacks like **AGC Scaling** drift metrics slowly over time, making instantaneous classification difficult. 

We use an **LSTM (Long Short-Term Memory)** network to analyze sequential time-windows of data.

In [ ]:
# 4A: Sequence Generation (Sliding Window)
SEQ_LEN = 10

def create_sequences(data, labels, seq_length):
    X_seq, y_seq = [], []
    for i in range(len(data) - seq_length):
        X_seq.append(data[i:(i + seq_length)])
        # The label for the sequence is the label of the last step
        y_seq.append(labels[i + seq_length - 1])
    return np.array(X_seq), np.array(y_seq)

# Must sort data logically for sequences. For simplicity, we use the un-shuffled scaled data.
X_scaled_full = scaler.fit_transform(X)
X_seq, y_seq = create_sequences(X_scaled_full, y, SEQ_LEN)

X_train_seq, X_test_seq, y_train_seq, y_test_seq = train_test_split(X_seq, y_seq, test_size=0.2, random_state=42)

train_tensor = TensorDataset(torch.FloatTensor(X_train_seq), torch.FloatTensor(y_train_seq).unsqueeze(1))
test_tensor = TensorDataset(torch.FloatTensor(X_test_seq), torch.FloatTensor(y_test_seq).unsqueeze(1))

train_loader = DataLoader(train_tensor, batch_size=64, shuffle=True)
test_loader = DataLoader(test_tensor, batch_size=64, shuffle=False)

print(f"Sequence Input Shape: {X_train_seq.shape} (Samples, Seq_Len, Features)")

In [ ]:
# 4B: LSTM Architecture Definition
class LSTMFraudDetector(nn.Module):
    def __init__(self, input_dim=20, hidden_dim=64, num_layers=2):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_dim, 
            hidden_size=hidden_dim, 
            num_layers=num_layers, 
            batch_first=True, 
            dropout=0.2
        )
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )
        
    def forward(self, x):
        # x shape: (batch_size, seq_len, input_dim)
        lstm_out, _ = self.lstm(x)
        # Extract the features of the last time step
        last_step = lstm_out[:, -1, :]
        return self.fc(last_step)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = LSTMFraudDetector().to(device)
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.005)

In [ ]:
# 4C: Training Loop
EPOCHS = 15
epoch_losses = []
val_losses = []

print(f"Training LSTM on {device}...")
for epoch in range(EPOCHS):
    model.train()
    batch_losses = []
    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        
        optimizer.zero_grad()
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        batch_losses.append(loss.item())
        
    epoch_losses.append(np.mean(batch_losses))
    
    # Validation Evaluation
    model.eval()
    val_batch_losses = []
    with torch.no_grad():
        for batch_x, batch_y in test_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            outputs = model(batch_x)
            val_batch_losses.append(criterion(outputs, batch_y).item())
    val_losses.append(np.mean(val_batch_losses))
    if (epoch+1) % 5 == 0:
        print(f'Epoch [{epoch+1}/{EPOCHS}], Train Loss: {epoch_losses[-1]:.4f}, Val Loss: {val_losses[-1]:.4f}')

In [ ]:
# Plot Learning Curve
plt.figure(figsize=(10, 5))
plt.plot(range(1, EPOCHS+1), epoch_losses, label='Training Loss', color='indigo', lw=2, marker='o')
plt.plot(range(1, EPOCHS+1), val_losses, label='Validation Loss', color='crimson', lw=2, marker='x', linestyle='--')
plt.title('LSTM Training & Validation Loss (BCE)')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# 4D: LSTM Evaluation on Sequence Test Set
model.eval()
lstm_preds = []
lstm_truths = []
with torch.no_grad():
    for batch_x, batch_y in test_loader:
        batch_x = batch_x.to(device)
        preds = model(batch_x).cpu().numpy()
        lstm_preds.extend(preds)
        lstm_truths.extend(batch_y.numpy())

lstm_preds = np.array(lstm_preds)
lstm_truths = np.array(lstm_truths)
y_pred_lstm = (lstm_preds > 0.5).astype(int)

print("LSTM Sequence Classification Report:\n")
print(classification_report(lstm_truths, y_pred_lstm))